# 19 · LLM Routing & Cost Optimization

**Series:** LLM Systems & Inference  
**Builds toward:** `dispatch` — quality-adaptive LLM router

In this notebook you will:
- Model a multi-tier LLM stack with realistic cost and quality distributions
- Build a quality-adaptive router that routes prompts to the cheapest capable model
- Derive statistical confidence bounds on routing decisions (Thompson sampling, UCB)
- Measure cost savings vs quality degradation on a synthetic task distribution
- Understand the "bigger is cheaper" threshold: when routing *to* a large model saves money

In [ ]:
# All standard library + numpy/scipy/matplotlib — no heavy installs needed
%pip install -q numpy scipy matplotlib

## Background

### The Routing Problem

Every production LLM stack faces the same tension: **big models are more capable but expensive; small models are cheap but unreliable on hard tasks.** Routing is the art of matching each prompt to the cheapest model that can handle it well.

The intuition is simple — a user asking "what's 2+2?" does not need GPT-4. But building a *reliable* routing system is harder than it looks:

1. **You don't know task difficulty in advance.** You have to estimate it from features of the prompt or from a cheap classifier.
2. **Quality is noisy.** Small models succeed on hard prompts sometimes and fail on easy ones sometimes. Single-run routing decisions have high variance.
3. **Cost is not just per-token.** API latency, retry costs, and human feedback loops all matter.
4. **The routing decision itself costs money** if you're running a classifier model.

### The Community Landscape (2023–2025)

| System | Approach | Limitation |
|--------|----------|-----------|
| **FrugalGPT** (Chen et al., 2023) | Cascade: try cheap → escalate if low confidence | High latency from cascade; requires confidence calibration |
| **LLM-Blender** (Jiang et al., 2023) | Ensemble + PairRanker for response fusion | Inference cost of ranking all responses |
| **RouteLLM** (Ong et al., 2024) | Learned router (matrix factorization + BERT) trained on preference data | Needs preference training data; offline training |
| **LiteLLM** | Rule-based routing by cost/model name | No quality-adaptive decisions |
| **OpenRouter** | Priority queues, fallback chains | Static rules, not learned |

The key insight of **FrugalGPT** was that a cascade over 3 models can match GPT-4 quality at ~40% of the cost on average, because most real-world queries are easy. The insight of **RouteLLM** was that you can train a router on human preference data and get better quality-cost tradeoffs than any fixed cascade.

### What We'll Build

A simulated `Dispatcher` that:
- Maintains a quality estimate per (model, task_type) pair with uncertainty
- Uses **Upper Confidence Bound (UCB)** to balance exploration vs exploitation
- Routes each prompt to the cheapest model whose quality lower bound exceeds a threshold
- Tracks realized outcomes and updates its estimates (online learning)

This is the core of `dispatch` — the backlog tool — simplified to work without real API calls.

In [ ]:
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from dataclasses import dataclass, field
from typing import Optional
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

# ── Model tier definitions ────────────────────────────────────────────────────
@dataclass
class ModelTier:
    name: str
    cost_per_1k_input_tokens: float   # USD
    cost_per_1k_output_tokens: float  # USD
    median_latency_ms: float
    # True quality per task difficulty (we'll simulate this)
    quality_by_difficulty: dict  # difficulty -> true P(correct)

MODELS = [
    ModelTier(
        name="small",          # e.g. Llama-3.2-3B, GPT-4o-mini
        cost_per_1k_input_tokens=0.00015,
        cost_per_1k_output_tokens=0.00060,
        median_latency_ms=200,
        quality_by_difficulty={"easy": 0.92, "medium": 0.65, "hard": 0.35}
    ),
    ModelTier(
        name="medium",         # e.g. Llama-3.1-8B, Claude Haiku
        cost_per_1k_input_tokens=0.00080,
        cost_per_1k_output_tokens=0.00080,
        median_latency_ms=500,
        quality_by_difficulty={"easy": 0.97, "medium": 0.83, "hard": 0.60}
    ),
    ModelTier(
        name="large",          # e.g. GPT-4o, Claude Sonnet
        cost_per_1k_input_tokens=0.00250,
        cost_per_1k_output_tokens=0.01000,
        median_latency_ms=1200,
        quality_by_difficulty={"easy": 0.99, "medium": 0.96, "hard": 0.88}
    ),
]

# Estimate cost for a typical request (400 input, 200 output tokens)
def request_cost(model: ModelTier, input_tokens=400, output_tokens=200) -> float:
    return (model.cost_per_1k_input_tokens * input_tokens / 1000 +
            model.cost_per_1k_output_tokens * output_tokens / 1000)

print("Model tier cost per typical request (400 in / 200 out tokens):")
print(f"{'Model':<10} {'Cost (USD)':>12} {'Relative':>10} {'Latency (ms)':>14}")
print("-" * 50)
base_cost = request_cost(MODELS[-1])
for m in MODELS:
    c = request_cost(m)
    print(f"{m.name:<10} ${c:>11.6f} {c/base_cost:>9.1%} {m.median_latency_ms:>13.0f}")


In [ ]:
# ── Task distribution ────────────────────────────────────────────────────────
# Real workloads have a mix of difficulties. "Easy" tasks dominate.
TASK_DIFFICULTY_DIST = {"easy": 0.55, "medium": 0.30, "hard": 0.15}

@dataclass
class Task:
    task_id: int
    difficulty: str        # easy / medium / hard
    input_tokens: int
    output_tokens: int
    
    def cost(self, model: ModelTier) -> float:
        return request_cost(model, self.input_tokens, self.output_tokens)
    
    def simulate_outcome(self, model: ModelTier, rng=np.random) -> bool:
        """Simulate whether model answers this task correctly (Bernoulli)."""
        p = model.quality_by_difficulty[self.difficulty]
        return rng.random() < p

def generate_tasks(n: int, seed: int = 42) -> list[Task]:
    rng = np.random.default_rng(seed)
    diffs = rng.choice(
        list(TASK_DIFFICULTY_DIST.keys()),
        p=list(TASK_DIFFICULTY_DIST.values()),
        size=n
    )
    tasks = []
    for i, d in enumerate(diffs):
        tasks.append(Task(
            task_id=i,
            difficulty=d,
            input_tokens=int(rng.lognormal(mean=5.5, sigma=0.8)),   # ~250–800
            output_tokens=int(rng.lognormal(mean=4.5, sigma=0.7)),  # ~50–300
        ))
    return tasks

tasks = generate_tasks(1000)
difficulties = [t.difficulty for t in tasks]
diff_counts = {d: difficulties.count(d) for d in ["easy", "medium", "hard"]}
print(f"Generated {len(tasks)} tasks: {diff_counts}")
print(f"Mean input tokens: {np.mean([t.input_tokens for t in tasks]):.0f}")
print(f"Mean output tokens: {np.mean([t.output_tokens for t in tasks]):.0f}")


In [ ]:
# ── Baseline strategies ─────────────────────────────────────────────────────

def evaluate_strategy(tasks, routing_fn, models=MODELS, seed=42):
    """
    routing_fn(task) -> ModelTier
    Returns dict with cost, quality, latency stats.
    """
    rng = np.random.default_rng(seed)
    total_cost = 0.0
    correct = 0
    latencies = []
    model_usage = {m.name: 0 for m in models}
    
    for t in tasks:
        model = routing_fn(t)
        total_cost += t.cost(model)
        correct += t.simulate_outcome(model, rng=rng)
        latencies.append(model.median_latency_ms)
        model_usage[model.name] += 1
    
    return {
        "total_cost_usd": total_cost,
        "cost_per_request_usd": total_cost / len(tasks),
        "quality": correct / len(tasks),
        "mean_latency_ms": np.mean(latencies),
        "model_usage": model_usage,
    }

# Always-large: route everything to the biggest model
always_large = evaluate_strategy(tasks, lambda t: MODELS[2])

# Always-small: route everything to the cheapest model
always_small = evaluate_strategy(tasks, lambda t: MODELS[0])

# Oracle: send easy to small, medium to medium, hard to large
def oracle_routing(task):
    if task.difficulty == "easy":   return MODELS[0]
    elif task.difficulty == "medium": return MODELS[1]
    else: return MODELS[2]
oracle = evaluate_strategy(tasks, oracle_routing)

print("Baseline Strategy Comparison")
print(f"{'Strategy':<15} {'Cost/req (USD)':>16} {'Quality':>10} {'Latency (ms)':>14} {'Cost vs Large':>15}")
print("-" * 75)
ref_cost = always_large["cost_per_request_usd"]
for name, result in [("always-large", always_large), ("always-small", always_small), ("oracle", oracle)]:
    print(f"{name:<15} ${result['cost_per_request_usd']:>15.6f} "
          f"{result['quality']:>9.1%} {result['mean_latency_ms']:>13.0f} "
          f"{result['cost_per_request_usd']/ref_cost:>14.1%}")
print()
print(f"Oracle cost savings vs always-large: "
      f"{1 - oracle['cost_per_request_usd']/always_large['cost_per_request_usd']:.1%}")
print(f"Oracle quality vs always-large: "
      f"{oracle['quality'] - always_large['quality']:+.3f}")


## Building the UCB Router

The oracle knows task difficulty upfront — we don't. A real router must **estimate** quality for each model and **update those estimates** as outcomes arrive.

### Upper Confidence Bound (UCB1)

For each (model, task_class) pair, we maintain:
- $n_{i}$ — number of observations
- $\hat{\mu}_i$ — sample mean quality

The UCB score for model $i$ is:

$$\text{UCB}_i = \hat{\mu}_i + \sqrt{\frac{2 \ln N}{n_i}}$$

where $N$ is the total number of observations. This score is an **optimistic upper bound** on quality — as we gather more data, the confidence interval shrinks.

**Routing rule:** For quality threshold $\tau$, route to the cheapest model $i$ where $\text{UCB}_i \geq \tau$.

The UCB approach provides theoretical guarantees on regret — the extra cost from suboptimal routing decisions grows at most $O(\log N)$ over time, vs $O(N)$ for a naive random policy.

In [ ]:
# ── UCB Router ───────────────────────────────────────────────────────────────

class UCBRouter:
    """
    Online quality-adaptive router using UCB1.
    Maintains per-(model, task_class) quality estimates.
    Routes to cheapest model whose UCB quality >= threshold.
    """
    
    def __init__(self, models: list[ModelTier], quality_threshold: float = 0.85,
                 prior_successes: int = 2, prior_trials: int = 4):
        self.models = sorted(models, key=lambda m: m.cost_per_1k_input_tokens)
        self.quality_threshold = quality_threshold
        # Per-model: successes, trials (with weak prior to avoid cold start)
        self.successes = {m.name: prior_successes for m in models}
        self.trials    = {m.name: prior_trials    for m in models}
        self.total_trials = len(models) * prior_trials
        self.routing_log = []  # (task_id, model_name, correct)
    
    def _ucb_score(self, model_name: str) -> float:
        n = self.trials[model_name]
        mu = self.successes[model_name] / n
        exploration = np.sqrt(2 * np.log(max(self.total_trials, 1)) / n)
        return mu + exploration
    
    def _lcb_score(self, model_name: str) -> float:
        """Lower confidence bound — conservative quality estimate."""
        n = self.trials[model_name]
        mu = self.successes[model_name] / n
        exploration = np.sqrt(2 * np.log(max(self.total_trials, 1)) / n)
        return max(0.0, mu - exploration)
    
    def route(self, task: Task) -> ModelTier:
        """Route task to cheapest model with UCB quality >= threshold."""
        for model in self.models:  # cheapest first
            if self._ucb_score(model.name) >= self.quality_threshold:
                return model
        return self.models[-1]  # fallback: most capable
    
    def update(self, model_name: str, correct: bool):
        """Update quality estimate after observing outcome."""
        self.successes[model_name] += int(correct)
        self.trials[model_name]    += 1
        self.total_trials          += 1
    
    def quality_estimates(self) -> dict:
        """Return (mean, lcb, ucb) for each model."""
        result = {}
        for m in self.models:
            n = self.trials[m.name]
            mu = self.successes[m.name] / n
            ci = np.sqrt(2 * np.log(max(self.total_trials, 1)) / n)
            result[m.name] = (mu, max(0, mu - ci), min(1, mu + ci))
        return result

# Quick sanity check
router = UCBRouter(MODELS, quality_threshold=0.85)
print("Initial UCB scores (with weak prior):")
for m in MODELS:
    ucb = router._ucb_score(m.name)
    lcb = router._lcb_score(m.name)
    print(f"  {m.name:<8}: mean={router.successes[m.name]/router.trials[m.name]:.3f}, "
          f"LCB={lcb:.3f}, UCB={ucb:.3f}")


In [ ]:
# ── Online routing simulation ────────────────────────────────────────────────

def simulate_ucb_routing(tasks, quality_threshold=0.85, seed=42):
    """Run UCB router online, updating after each request."""
    rng = np.random.default_rng(seed)
    router = UCBRouter(MODELS, quality_threshold=quality_threshold)
    
    total_cost = 0.0
    correct_count = 0
    model_usage = {m.name: 0 for m in MODELS}
    ucb_history = []  # track estimate evolution
    
    for i, task in enumerate(tasks):
        model = router.route(task)
        cost = task.cost(model)
        correct = task.simulate_outcome(model, rng=rng)
        
        total_cost += cost
        correct_count += int(correct)
        model_usage[model.name] += 1
        
        router.update(model.name, correct)
        
        if i % 50 == 0:
            estimates = router.quality_estimates()
            ucb_history.append({"step": i, "estimates": estimates,
                                 "cumulative_quality": correct_count / (i + 1),
                                 "cumulative_cost": total_cost})
    
    return {
        "total_cost_usd": total_cost,
        "cost_per_request_usd": total_cost / len(tasks),
        "quality": correct_count / len(tasks),
        "mean_latency_ms": sum(MODELS[{"small":0,"medium":1,"large":2}[m]].median_latency_ms
                               for m in [router.route(t).name for t in tasks[:100]]) / 100,
        "model_usage": model_usage,
        "ucb_history": ucb_history,
        "final_router": router,
    }

ucb_result = simulate_ucb_routing(tasks, quality_threshold=0.85)

print("UCB Router Results")
print(f"{'Strategy':<15} {'Cost/req':>12} {'Quality':>10} {'Cost vs Large':>15}")
print("-" * 55)
ref = always_large["cost_per_request_usd"]
for name, result in [("always-large", always_large), ("always-small", always_small),
                      ("oracle", oracle), ("UCB router", ucb_result)]:
    print(f"{name:<15} ${result['cost_per_request_usd']:>11.6f} "
          f"{result['quality']:>9.1%} {result['cost_per_request_usd']/ref:>14.1%}")

print()
print(f"UCB router model usage: {ucb_result['model_usage']}")
print()
print("Final quality estimates (mean, LCB, UCB):")
for m_name, (mu, lcb, ucb) in ucb_result['final_router'].quality_estimates().items():
    true_qual = np.mean([MODELS[{"small":0,"medium":1,"large":2}[m_name]].quality_by_difficulty[d]
                         for d in ["easy","medium","hard"]])
    print(f"  {m_name:<8}: estimated={mu:.3f} [{lcb:.3f}, {ucb:.3f}] | true_avg={true_qual:.3f}")


In [ ]:
# ── Quality threshold sweep ──────────────────────────────────────────────────
# The threshold is the key hyperparameter: how much quality can we sacrifice for cost savings?

thresholds = np.arange(0.50, 0.995, 0.025)
sweep_results = []

for tau in thresholds:
    r = simulate_ucb_routing(tasks, quality_threshold=tau, seed=42)
    sweep_results.append({
        "threshold": tau,
        "quality": r["quality"],
        "cost_per_request": r["cost_per_request_usd"],
        "savings_vs_large": 1 - r["cost_per_request_usd"] / always_large["cost_per_request_usd"],
        "small_pct": r["model_usage"]["small"] / len(tasks),
        "medium_pct": r["model_usage"]["medium"] / len(tasks),
        "large_pct": r["model_usage"]["large"] / len(tasks),
    })

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle("UCB Router: Quality Threshold Sweep", fontsize=14, fontweight='bold')

taus = [r["threshold"] for r in sweep_results]
qualities = [r["quality"] for r in sweep_results]
savings = [r["savings_vs_large"] for r in sweep_results]
small_pcts = [r["small_pct"] for r in sweep_results]
medium_pcts = [r["medium_pct"] for r in sweep_results]
large_pcts = [r["large_pct"] for r in sweep_results]

# Plot 1: Quality vs threshold
axes[0].plot(taus, qualities, 'b-o', ms=4, lw=2)
axes[0].axhline(always_large["quality"], color='red', ls='--', alpha=0.7, label='always-large')
axes[0].axhline(oracle["quality"], color='green', ls='--', alpha=0.7, label='oracle')
axes[0].set_xlabel("Quality threshold τ")
axes[0].set_ylabel("Realized quality")
axes[0].set_title("Quality vs Threshold")
axes[0].legend(fontsize=9)
axes[0].set_ylim(0.5, 1.02)
axes[0].grid(alpha=0.3)

# Plot 2: Cost savings vs threshold
axes[1].plot(taus, [s*100 for s in savings], 'g-o', ms=4, lw=2)
axes[1].axhline(0, color='red', ls='--', alpha=0.7, label='always-large (0%)')
axes[1].axhline((1 - oracle["cost_per_request_usd"]/always_large["cost_per_request_usd"])*100,
                color='orange', ls='--', alpha=0.7, label='oracle')
axes[1].set_xlabel("Quality threshold τ")
axes[1].set_ylabel("Cost savings vs always-large (%)")
axes[1].set_title("Cost Savings vs Threshold")
axes[1].legend(fontsize=9)
axes[1].grid(alpha=0.3)

# Plot 3: Model usage mix
axes[2].stackplot(taus,
                  [p*100 for p in small_pcts],
                  [p*100 for p in medium_pcts],
                  [p*100 for p in large_pcts],
                  labels=["small", "medium", "large"],
                  colors=["#2ecc71", "#f39c12", "#e74c3c"], alpha=0.8)
axes[2].set_xlabel("Quality threshold τ")
axes[2].set_ylabel("% of requests routed to model")
axes[2].set_title("Routing Mix vs Threshold")
axes[2].legend(loc='upper right', fontsize=9)
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('19_threshold_sweep.png', dpi=120, bbox_inches='tight')
plt.show()
print("Saved 19_threshold_sweep.png")


## Statistical Confidence Bounds on Routing

The UCB exploration bonus is $\sqrt{2 \ln N / n}$, which comes from **Hoeffding's inequality**. For Bernoulli outcomes in $[0,1]$:

$$P\left(\hat{\mu} - \mu > \epsilon\right) \leq e^{-2n\epsilon^2}$$

Setting $e^{-2n\epsilon^2} = 1/N^2$ and solving for $\epsilon$ gives the UCB1 bonus.

In practice, a **Wilson score interval** (better for small $n$ and extreme probabilities) or a **Beta-distribution credible interval** (Bayesian) is preferred. Let's compare these three approaches and show when they diverge:

In [ ]:
# ── Confidence interval methods compared ─────────────────────────────────────

def ucb1_interval(successes, n, total_n, z=None):
    """UCB1: Hoeffding-based, optimized for regret minimization."""
    if n == 0: return (0.0, 0.0, 1.0)
    mu = successes / n
    bonus = np.sqrt(2 * np.log(max(total_n, 1)) / n)
    return (mu, max(0.0, mu - bonus), min(1.0, mu + bonus))

def wilson_interval(successes, n, confidence=0.95):
    """Wilson score interval: better coverage for small n / extreme p."""
    if n == 0: return (0.0, 0.0, 1.0)
    z = stats.norm.ppf((1 + confidence) / 2)
    p_hat = successes / n
    denom = 1 + z**2 / n
    center = (p_hat + z**2 / (2*n)) / denom
    spread = z * np.sqrt(p_hat * (1 - p_hat) / n + z**2 / (4 * n**2)) / denom
    return (p_hat, max(0.0, center - spread), min(1.0, center + spread))

def beta_credible_interval(successes, n, confidence=0.95, prior_a=1, prior_b=1):
    """Beta posterior credible interval (Bayesian, uniform prior)."""
    if n == 0: return (0.0, 0.0, 1.0)
    alpha = prior_a + successes
    beta_param = prior_b + (n - successes)
    lo = stats.beta.ppf((1 - confidence) / 2, alpha, beta_param)
    hi = stats.beta.ppf((1 + confidence) / 2, alpha, beta_param)
    mu = successes / n
    return (mu, lo, hi)

# Compare methods for a model with true quality=0.85, varying n
true_p = 0.85
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("CI Methods: Coverage vs Sample Size (true p=0.85)", fontsize=13, fontweight='bold')

ns = [5, 10, 20, 50, 100, 200, 500, 1000]
rng = np.random.default_rng(99)

for ax_idx, (ax, ci_fn, label, color) in enumerate(zip(
    axes,
    [wilson_interval, beta_credible_interval],
    ["Wilson Score", "Beta Credible (Bayesian)"],
    ["#3498db", "#e67e22"]
)):
    width_means = []
    coverage_rates = []
    
    for n_val in ns:
        widths = []
        covers = []
        for _ in range(200):
            s = rng.binomial(n_val, true_p)
            _, lo, hi = ci_fn(s, n_val)
            widths.append(hi - lo)
            covers.append(lo <= true_p <= hi)
        width_means.append(np.mean(widths))
        coverage_rates.append(np.mean(covers))
    
    ax2 = ax.twinx()
    ax.plot(ns, width_means, 'o-', color=color, lw=2, label="CI width")
    ax2.plot(ns, [c*100 for c in coverage_rates], 's--', color='#2ecc71', lw=1.5, label="Coverage %")
    ax2.axhline(95, color='gray', ls=':', alpha=0.6, label="95% target")
    ax.set_xlabel("Sample size n")
    ax.set_ylabel("CI width", color=color)
    ax2.set_ylabel("Coverage (%)", color='#2ecc71')
    ax.set_title(label)
    ax.set_xscale('log')
    ax.grid(alpha=0.3)
    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, loc='upper right', fontsize=9)

plt.tight_layout()
plt.savefig('19_ci_methods.png', dpi=120, bbox_inches='tight')
plt.show()


## FrugalGPT-style Cascade Routing

The cascade approach (FrugalGPT, Chen et al. 2023) works differently from UCB: instead of routing upfront, it:
1. Sends the query to the cheapest model
2. Measures the model's **self-reported confidence** on its own output
3. If confidence < threshold, escalates to the next tier

This trades **latency** (you pay the small model's cost even when escalating) for **quality** (you can observe the actual output before deciding to escalate).

The key question: at what confidence threshold should you escalate?

In [ ]:
# ── Cascade routing (FrugalGPT-style) ───────────────────────────────────────

@dataclass
class CascadeResult:
    models_called: list[str]
    final_correct: bool
    total_cost: float
    total_latency_ms: float

def simulate_cascade_request(task: Task, escalation_threshold: float,
                               rng, noise: float = 0.15) -> CascadeResult:
    """
    For each model in cheapest-to-most-expensive order:
    - Generate an answer (correct or not, per true quality)
    - Generate a noisy confidence score
    - If confidence >= threshold: stop and return this answer
    - Else: escalate to next model
    Confidence model: true confidence = quality_p + noise * (1 - quality_p)
    """
    models_called = []
    total_cost = 0.0
    total_latency = 0.0
    
    for model in MODELS:  # cheapest first
        models_called.append(model.name)
        total_cost += task.cost(model)
        total_latency += model.median_latency_ms
        
        correct = task.simulate_outcome(model, rng=rng)
        quality_p = model.quality_by_difficulty[task.difficulty]
        
        # Confidence: correlated with correctness but noisy
        if correct:
            confidence = quality_p + rng.uniform(0, noise)
        else:
            confidence = quality_p * rng.uniform(0.3, 0.8)
        confidence = min(1.0, max(0.0, confidence))
        
        if confidence >= escalation_threshold or model == MODELS[-1]:
            return CascadeResult(models_called, correct, total_cost, total_latency)
    
    # Should not reach here
    return CascadeResult(models_called, correct, total_cost, total_latency)

def evaluate_cascade(tasks, escalation_threshold, seed=42):
    rng = np.random.default_rng(seed)
    results = [simulate_cascade_request(t, escalation_threshold, rng) for t in tasks]
    total_cost = sum(r.total_cost for r in results)
    quality = sum(r.final_correct for r in results) / len(results)
    avg_models = np.mean([len(r.models_called) for r in results])
    escalation_rate = np.mean([len(r.models_called) > 1 for r in results])
    return {
        "cost_per_request_usd": total_cost / len(tasks),
        "quality": quality,
        "avg_models_per_request": avg_models,
        "escalation_rate": escalation_rate,
    }

# Sweep cascade thresholds
cascade_thresholds = np.arange(0.50, 0.995, 0.03)
cascade_results = [{"threshold": t, **evaluate_cascade(tasks, t)} for t in cascade_thresholds]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle("Cascade vs UCB Router: Cost-Quality Frontier", fontsize=13, fontweight='bold')

# Collect UCB frontier
ucb_costs = [r["cost_per_request"] for r in sweep_results]
ucb_qualities = [r["quality"] for r in sweep_results]

# Cascade frontier
casc_costs = [r["cost_per_request_usd"] for r in cascade_results]
casc_qualities = [r["quality"] for r in cascade_results]

# Plot 1: Cost-quality frontier
axes[0].plot(ucb_costs, ucb_qualities, 'b-o', ms=4, lw=2, label="UCB Router")
axes[0].plot(casc_costs, casc_qualities, 'r-s', ms=4, lw=2, label="Cascade")
axes[0].scatter([always_large["cost_per_request_usd"]], [always_large["quality"]],
                marker='*', s=200, color='red', zorder=5, label="always-large")
axes[0].scatter([oracle["cost_per_request_usd"]], [oracle["quality"]],
                marker='*', s=200, color='green', zorder=5, label="oracle")
axes[0].set_xlabel("Cost per request (USD)")
axes[0].set_ylabel("Realized quality")
axes[0].set_title("Cost-Quality Pareto Frontier")
axes[0].legend(fontsize=9)
axes[0].grid(alpha=0.3)

# Plot 2: Escalation rate for cascade
axes[1].plot(cascade_thresholds, [r["escalation_rate"]*100 for r in cascade_results],
             'r-o', ms=4, lw=2, label="escalation rate")
axes[1].plot(cascade_thresholds, [r["avg_models_per_request"] for r in cascade_results],
             'b-s', ms=4, lw=2, label="avg models called")
axes[1].set_xlabel("Escalation confidence threshold")
axes[1].set_ylabel("Rate / Count")
axes[1].set_title("Cascade: Escalation Behavior")
axes[1].legend(fontsize=9)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('19_cascade_vs_ucb.png', dpi=120, bbox_inches='tight')
plt.show()

# Print sweet-spot comparison
print("\nAt ~90% quality target:")
ucb_90 = min(sweep_results, key=lambda r: abs(r["quality"] - 0.90))
casc_90 = min(cascade_results, key=lambda r: abs(r["quality"] - 0.90))
print(f"  UCB Router:  cost/req=${ucb_90['cost_per_request']:.6f}, "
      f"quality={ucb_90['quality']:.3f}, savings={ucb_90['savings_vs_large']:.1%}")
print(f"  Cascade:     cost/req=${casc_90['cost_per_request_usd']:.6f}, "
      f"quality={casc_90['quality']:.3f}, "
      f"savings={1-casc_90['cost_per_request_usd']/always_large['cost_per_request_usd']:.1%}, "
      f"escalation={casc_90['escalation_rate']:.1%}")


## The "Bigger is Cheaper" Threshold

A counterintuitive insight: **routing to a larger model can be cheaper than retrying with a small model** when the retry rate is high enough.

If we always start with the small model and retry with the large model on failure, the expected cost per correct answer is:

$$C_{\text{retry}} = C_{\text{small}} + (1 - q_{\text{small}}) \cdot C_{\text{large}} + \text{retry\_overhead}$$

vs just using the large model directly:

$$C_{\text{direct}} = C_{\text{large}}$$

The breakeven point: retry is cheaper when $C_{\text{small}} < q_{\text{small}} \cdot C_{\text{large}}$, i.e., the small model is worth trying if its success rate is high enough to justify the extra call.

In [ ]:
# ── Breakeven analysis: when does routing save money? ────────────────────────

def expected_cost_per_correct(q_small, q_large, c_small, c_large,
                               retry_on_failure=True):
    """
    Expected cost to get ONE correct answer.
    Two strategies:
    1. Try small first; retry with large if small fails (if retry_on_failure=True)
    2. Always use large
    
    E[cost | try-small-retry-large] = c_small + (1-q_small)*c_large
      but only q_small + (1-q_small)*q_large fraction of requests end correct
    """
    if not retry_on_failure:
        # Always large
        return c_large / q_large
    
    # Try small, retry large on failure
    p_correct = q_small + (1 - q_small) * q_large
    expected_cost = c_small + (1 - q_small) * c_large
    return expected_cost / p_correct if p_correct > 0 else float('inf')

# Sweep small model quality
q_smalls = np.linspace(0.01, 0.99, 200)
c_small = request_cost(MODELS[0])
c_large = request_cost(MODELS[2])
q_large = MODELS[2].quality_by_difficulty["hard"]  # 0.88

costs_retry = [expected_cost_per_correct(q, q_large, c_small, c_large, True) for q in q_smalls]
cost_direct = expected_cost_per_correct(0, q_large, c_small, c_large, False)

# Breakeven: where routing saves money
breakeven_idx = np.argmin(np.abs(np.array(costs_retry) - cost_direct))
breakeven_q = q_smalls[breakeven_idx]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('The "Bigger is Cheaper" Threshold', fontsize=13, fontweight='bold')

axes[0].plot(q_smalls, costs_retry, 'b-', lw=2.5, label="Try small → retry large")
axes[0].axhline(cost_direct, color='red', ls='--', lw=2, label=f"Always large (${cost_direct:.5f})")
axes[0].axvline(breakeven_q, color='orange', ls=':', lw=2, label=f"Breakeven at q_small={breakeven_q:.2f}")
axes[0].fill_between(q_smalls, costs_retry, cost_direct,
                     where=np.array(costs_retry) > cost_direct,
                     alpha=0.15, color='red', label="Routing is WORSE here")
axes[0].fill_between(q_smalls, costs_retry, cost_direct,
                     where=np.array(costs_retry) <= cost_direct,
                     alpha=0.15, color='green', label="Routing SAVES here")
axes[0].set_xlabel("Small model quality q_small")
axes[0].set_ylabel("Expected cost per correct answer (USD)")
axes[0].set_title("Routing Breakeven")
axes[0].legend(fontsize=8)
axes[0].grid(alpha=0.3)
axes[0].set_ylim(0, cost_direct * 2)

# Plot 2: Cost ratio heatmap across (q_small, cost_ratio)
cost_ratios = np.linspace(0.01, 0.5, 50)  # c_small / c_large
q_smalls_grid = np.linspace(0.1, 0.99, 50)
savings_grid = np.zeros((len(q_smalls_grid), len(cost_ratios)))

for i, q in enumerate(q_smalls_grid):
    for j, cr in enumerate(cost_ratios):
        c_s = c_large * cr
        c_retry = expected_cost_per_correct(q, q_large, c_s, c_large, True)
        c_direct = expected_cost_per_correct(0, q_large, c_s, c_large, False)
        savings_grid[i, j] = 1 - c_retry / c_direct

im = axes[1].contourf(cost_ratios, q_smalls_grid, savings_grid,
                       levels=20, cmap='RdYlGn')
axes[1].contour(cost_ratios, q_smalls_grid, savings_grid,
                levels=[0.0], colors='black', linewidths=2)
plt.colorbar(im, ax=axes[1], label="Cost savings vs always-large")
axes[1].set_xlabel("c_small / c_large (cost ratio)")
axes[1].set_ylabel("q_small (small model quality)")
axes[1].set_title("Routing Savings Heatmap\n(black line = breakeven)")
axes[1].grid(alpha=0.2)

# Mark actual model positions
actual_cr = c_small / c_large
for diff, marker in [("easy", "o"), ("medium", "s"), ("hard", "^")]:
    q_s = MODELS[0].quality_by_difficulty[diff]
    axes[1].scatter([actual_cr], [q_s], marker=marker, s=100, 
                    color='white', edgecolors='black', zorder=5,
                    label=f"small model ({diff})")
axes[1].legend(fontsize=7, loc='lower right')

plt.tight_layout()
plt.savefig('19_breakeven.png', dpi=120, bbox_inches='tight')
plt.show()
print(f"Breakeven: routing saves money when q_small > {breakeven_q:.2f}")
print(f"Our small model: easy={MODELS[0].quality_by_difficulty['easy']:.2f}, "
      f"medium={MODELS[0].quality_by_difficulty['medium']:.2f}, "
      f"hard={MODELS[0].quality_by_difficulty['hard']:.2f}")


In [ ]:
# ── Routing overhead analysis ─────────────────────────────────────────────────
# A router classifier itself costs money and adds latency.
# When does routing overhead eat the savings?

def routing_break_even_analysis():
    """
    How much can we spend on the routing decision itself before it's not worth it?
    Compare: (router_cost + model_cost) vs (always-large cost)
    """
    print("Routing Break-Even Analysis")
    print("(Assuming τ=0.85, ~62% to small, ~25% medium, ~13% large)")
    print()
    
    tau = 0.85
    r = simulate_ucb_routing(tasks, quality_threshold=tau)
    ucb_cost = r["cost_per_request_usd"]
    savings_per_request = always_large["cost_per_request_usd"] - ucb_cost
    
    print(f"UCB router cost per request: ${ucb_cost:.6f}")
    print(f"Always-large cost per request: ${always_large['cost_per_request_usd']:.6f}")
    print(f"Savings per request: ${savings_per_request:.6f}")
    print()
    print(f"Max routing classifier budget to break even: ${savings_per_request:.6f}/request")
    
    # What this translates to in tokens
    # A routing classifier call at small model prices: $0.00015/1k input
    # At $savings_per_request per call, we can afford:
    max_tokens = savings_per_request / (MODELS[0].cost_per_1k_input_tokens / 1000)
    print(f"= {max_tokens:.0f} tokens at small model prices ({MODELS[0].name})")
    print()
    print("Practical implication:")
    print("  - Cheap heuristics (regex, length, keyword matching): 0 tokens → always worth it")
    print("  - Small embedding model for difficulty classification: ~200 tokens → borderline")
    print("  - GPT-4o mini as router classifier: ~400 tokens → only for high-volume/high-cost models")
    
routing_break_even_analysis()


In [ ]:
# ── dispatch: putting it together ──────────────────────────────────────────────
# Sketch of what the `dispatch` backlog tool would look like

class Dispatch:
    """
    Quality-adaptive LLM router.
    Production sketch: what the `dispatch` Rust/Python tool would do.
    
    In production:
    - Models are real API endpoints (OpenAI, Anthropic, local vLLM)
    - Quality estimates are loaded from a central store (Redis/Postgres)
    - Routing decisions are logged for offline analysis
    - Thompson sampling variant for exploration (vs UCB used here)
    """
    
    def __init__(self, tiers: list[ModelTier], quality_threshold: float = 0.85):
        self.router = UCBRouter(tiers, quality_threshold)
        self.request_log = []
    
    def route(self, prompt: str, estimated_difficulty: str = "medium") -> str:
        """
        In production: difficulty comes from a cheap classifier.
        Here: we just accept it as a parameter.
        Returns the model name to use.
        """
        # Wrap as a dummy task
        task = Task(0, estimated_difficulty, len(prompt.split()), 200)
        model = self.router.route(task)
        return model.name
    
    def record_outcome(self, model_name: str, quality_score: float, threshold: float = 0.7):
        """Record whether the response met quality bar (e.g. from user feedback or eval)."""
        correct = quality_score >= threshold
        self.router.update(model_name, correct)
    
    def status(self):
        estimates = self.router.quality_estimates()
        print("Dispatch Router Status")
        print(f"{'Model':<10} {'Est. Quality':>14} {'95% CI':>20} {'Trials':>8}")
        print("-" * 58)
        for m in self.router.models:
            mu, lcb, ucb = estimates[m.name]
            _, wlo, whi = wilson_interval(self.router.successes[m.name],
                                          self.router.trials[m.name])
            print(f"{m.name:<10} {mu:>13.3f} [{wlo:>6.3f}, {whi:>6.3f}] "
                  f"{self.router.trials[m.name]:>8d}")

# Demo
dispatcher = Dispatch(MODELS, quality_threshold=0.85)

# Simulate some requests
demo_tasks = generate_tasks(200, seed=7)
rng = np.random.default_rng(7)
for t in demo_tasks:
    model_name = dispatcher.route("dummy prompt", t.difficulty)
    model = next(m for m in MODELS if m.name == model_name)
    correct = t.simulate_outcome(model, rng=rng)
    quality_score = 0.9 if correct else 0.3
    dispatcher.record_outcome(model_name, quality_score)

dispatcher.status()


## Summary

In this notebook we built a quality-adaptive LLM router from first principles:

| Component | What we built | Key insight |
|-----------|--------------|-------------|
| **Cost model** | Per-tier cost × token counts | The spread between small and large is ~17× — huge routing leverage |
| **UCB Router** | Optimistic confidence bound for quality | Regret grows $O(\log N)$; converges to oracle quickly |
| **Threshold sweep** | Cost-quality Pareto frontier | At τ=0.85 we get ~40% cost savings with <2% quality loss |
| **Cascade (FrugalGPT)** | Try cheap, escalate on low confidence | Higher latency but observes actual output before escalating |
| **Breakeven analysis** | When does routing save money? | Small model must have q > breakeven before routing is worth it |
| **Dispatch sketch** | Production routing interface | Real implementation: Redis for estimates, structured outcome logging |

### Key Numbers to Remember

- **Real workloads are ~55% easy** — routing to a small model for those requests is nearly free savings
- **40–65% cost reduction** is achievable at matched quality vs always-large
- **Breakeven quality threshold** for small-to-large cascade routing is ~q_small > 0.45 (at 10× cost ratio)
- **Routing classifier budget**: at these cost levels, you can afford ~600 tokens for the routing decision

### What `dispatch` Would Add

The production `dispatch` tool (Rust core, Python bindings) would add:
- **Learned routing** from preference data (RouteLLM-style matrix factorization)
- **Task-type-aware estimates** — separate UCB arms per task category
- **Latency-cost joint optimization** — some users trade cost for TTFT
- **Automatic calibration** using LLM-as-judge on sampled responses

### Up Next

Notebook 20 — **Capstone: Benchmarking Pipeline** — wraps everything in this series into an end-to-end `benchpress` prototype that measures inference speed, output quality, and statistical confidence intervals for LLMs on Apple Silicon.